# Entrenamiento Urban-v1 completo en GPU (A2D2 + UrbanSyn)

Notebook autorizada para iniciar los cuatro experimentos del protocolo. Recupera y valida metadatos TRAIN/VAL, estima almacenamiento, materializa los 3.751 registros, ejecuta un piloto de 10 batches por configuración y, si los cuatro pilotos pasan, entrena los modelos con checkpoints reanudables en Drive.

**No usa TEST.** Los datos se descargan directamente al almacenamiento temporal de Colab, nunca al teléfono. Los checkpoints, métricas y logs se guardan en Drive privado. Requiere una GPU CUDA asignada. La notebook se detiene ante falta de espacio, datos incompletos, OOM o métricas inválidas.

## 1. Montar Drive y sincronizar el repositorio privado

Añade un secreto de Colab llamado `GITHUB_TOKEN` con acceso de solo lectura a Contents y Actions para este repositorio.

In [ ]:
from google.colab import drive, userdata
from pathlib import Path
from getpass import getpass
import os, sys, json, hashlib, subprocess, shutil, zipfile, requests, datetime

drive.mount('/content/drive')
REPO = 'Leohf04/Procesamiento-de-Im-genes-'
BRANCH = 'urban-models-and-benchmark'
WORKDIR = Path('/content/Procesamiento-de-Im-genes-')

try:
    TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    TOKEN = None
if not TOKEN:
    TOKEN = getpass('GitHub PAT (Contents: Read-only + Actions: Read-only): ')
if not TOKEN:
    raise RuntimeError('Falta GITHUB_TOKEN.')

askpass = Path('/tmp/urban_v1_git_askpass.sh')
askpass.write_text('#!/bin/sh\ncase "$1" in\n *Username*) echo x-access-token ;;\n *Password*) echo "$COLAB_GITHUB_TOKEN" ;;\n *) exit 1 ;;\nesac\n')
askpass.chmod(0o700)
env = os.environ.copy()
env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'COLAB_GITHUB_TOKEN': TOKEN})
if WORKDIR.exists() and (WORKDIR/'.git').is_dir():
    subprocess.run(['git','-C',str(WORKDIR),'fetch','origin',BRANCH],check=True,env=env)
    subprocess.run(['git','-C',str(WORKDIR),'checkout',BRANCH],check=True)
    subprocess.run(['git','-C',str(WORKDIR),'pull','--ff-only','origin',BRANCH],check=True,env=env)
elif WORKDIR.exists():
    raise RuntimeError(f'{WORKDIR} existe pero no es un repo Git; no se sobrescribe.')
else:
    subprocess.run(['git','clone','--single-branch','--branch',BRANCH,
                    f'https://github.com/{REPO}.git',str(WORKDIR)],check=True,env=env)
del env
HEAD = subprocess.check_output(['git','-C',str(WORKDIR),'rev-parse','HEAD'],text=True).strip()
print('HEAD:', HEAD)
print('RUN_TRAINING=True (autorización del usuario recibida)')

## 1A. Verificar GPU antes de recuperar datos

En Colab selecciona **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU** antes de ejecutar. Si no hay GPU CUDA, esta notebook se detiene sin descargar datos.

In [ ]:
import torch, shutil, os, platform
if not torch.cuda.is_available():
    raise RuntimeError("GPU CUDA no asignada. Cambia el acelerador a T4 GPU y vuelve a ejecutar.")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_GIB = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print({"gpu": GPU_NAME, "vram_gib": round(GPU_VRAM_GIB, 2),
       "torch": torch.__version__, "cuda": torch.version.cuda,
       "free_disk_gib": round(shutil.disk_usage('/content').free/(1024**3), 2)})
if GPU_VRAM_GIB < 10:
    print("ADVERTENCIA: VRAM inferior a 10 GiB; el piloto puede detenerse por OOM.")

## 2. Recuperar y verificar el artifact

Se comprueban repositorio, run, caducidad y digest publicado por GitHub. No se extrae el ZIP completo.

In [ ]:
ARTIFACT_ID = 14770171764
RUN_ID = 45038801547
EXPECTED_SHA256 = 'b6cbcb54dfd0598e6a9011667073028be1804fc1c112d82e516a42bdbf2ba4e5'
API = f'https://api.github.com/repos/{REPO}/actions/artifacts/{ARTIFACT_ID}'
headers = {'Authorization': f'Bearer {TOKEN}', 'Accept': 'application/vnd.github+json',
           'X-GitHub-Api-Version': '2022-11-28'}
meta_response = requests.get(API, headers=headers, timeout=60)
if meta_response.status_code != 200:
    raise RuntimeError(f'No se pudo leer el artifact ({meta_response.status_code}): {meta_response.text[:700]}')
meta = meta_response.json()
if meta.get('expired'):
    raise RuntimeError('El artifact ha expirado.')
if meta.get('workflow_run',{}).get('id') != RUN_ID:
    raise RuntimeError('El artifact no pertenece al run esperado.')
if meta.get('digest') != f'sha256:{EXPECTED_SHA256}':
    raise RuntimeError(f'Digest publicado inesperado: {meta.get("digest")}')
zip_response = requests.get(API+'/zip', headers=headers, timeout=300)
if zip_response.status_code != 200:
    raise RuntimeError(f'Falló descarga del ZIP ({zip_response.status_code}): {zip_response.text[:500]}')
ZIP_PATH = Path('/content/urban_annotation_artifact_14770171764.zip')
ZIP_PATH.write_bytes(zip_response.content)
actual_sha = hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest()
if actual_sha != EXPECTED_SHA256:
    ZIP_PATH.unlink(missing_ok=True)
    raise RuntimeError(f'SHA-256 del ZIP no coincide: {actual_sha}')
print(json.dumps({'artifact_id':ARTIFACT_ID,'name':meta.get('name'),
                  'expired':meta.get('expired'),'size_in_bytes':len(zip_response.content),
                  'sha256':actual_sha,'expires_at':meta.get('expires_at')},indent=2))
del TOKEN, headers, meta_response, zip_response

## 3. Reconstruir metadatos completos TRAIN/VAL

Solo se abren los cuatro pares de manifiestos semánticos/instancias y los cuatro índices TRAIN/VAL. No se extrae ningún archivo TEST. Se validan filas, IDs, estadísticas y assets requeridos.

In [ ]:
import json, hashlib, zipfile
from pathlib import Path

def read_jsonl_bytes(raw):
    return [json.loads(line) for line in raw.decode('utf-8').splitlines() if line.strip()]

def write_jsonl(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w',encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row,sort_keys=True,ensure_ascii=False)+'\n')

def member_for(zf, suffix):
    hits = [n for n in zf.namelist() if n.endswith(suffix)]
    if len(hits) != 1:
        raise RuntimeError(f'Esperaba un único miembro para {suffix}; encontrados={len(hits)}')
    return hits[0]

EXPECTED = {('a2d2','train'):1000,('a2d2','val'):1000,
            ('urbansyn','train'):1000,('urbansyn','val'):751}
ASSET_KEYS = {'a2d2':('rgb','semantic'),'urbansyn':('rgb','semantic','panoptic')}
manifest_out = {}
index_out = {}
checks = []
with zipfile.ZipFile(ZIP_PATH,'r') as zf:
    for ds in ('a2d2','urbansyn'):
        for split in ('train','val'):
            stem = f'{ds}_{split}'
            sem_name = member_for(zf,f'balanced/{ds}_semantic_{split}.jsonl')
            ins_name = member_for(zf,f'balanced/{ds}_instances_{split}.jsonl')
            ann_name = member_for(zf,f'annotation_index/{ds}_{split}.annotations.jsonl')
            sem = read_jsonl_bytes(zf.read(sem_name))
            ins = read_jsonl_bytes(zf.read(ins_name))
            ann = read_jsonl_bytes(zf.read(ann_name))
            if len(sem) != EXPECTED[(ds,split)] or len(ins) != EXPECTED[(ds,split)] or len(ann) != EXPECTED[(ds,split)]:
                raise RuntimeError(f'{stem}: conteos inesperados sem/inst/index={len(sem)}/{len(ins)}/{len(ann)}; esperado={EXPECTED[(ds,split)]}')
            def keyed(rows, label):
                ids = [str(x.get('sample_id','')) for x in rows]
                if any(not x for x in ids) or len(ids) != len(set(ids)):
                    raise RuntimeError(f'{stem}: IDs vacíos o duplicados en {label}')
                return {str(x['sample_id']):x for x in rows}
            sm, im, am = keyed(sem,'semantic'), keyed(ins,'instances'), keyed(ann,'annotation_index')
            if set(sm) != set(im) or set(sm) != set(am):
                raise RuntimeError(f'{stem}: conjuntos de IDs no coinciden entre los tres archivos')
            for sid,row in sm.items():
                if any(row.get(k) != am[sid].get(k) for k in ('semantic_pixels','instance_counts')):
                    raise RuntimeError(f'{stem}: estadísticas no coinciden con índice para {sid}')
                if any(im[sid].get(k) != row.get(k) for k in ('semantic_pixels','instance_counts')):
                    raise RuntimeError(f'{stem}: estadísticas sem/inst distintas para {sid}')
                sem_base = {k:v for k,v in row.items() if k not in ('semantic_pixels','instance_counts')}
                ins_base = {k:v for k,v in im[sid].items() if k not in ('semantic_pixels','instance_counts')}
                if sem_base != ins_base:
                    raise RuntimeError(f'{stem}: metadatos sem/inst distintos para {sid}')
                if am[sid].get('annotation_status','ok') != 'ok':
                    raise RuntimeError(f'{stem}: annotation_status no OK para {sid}')
                assets = row.get('assets',{})
                for key in ASSET_KEYS[ds]:
                    if not isinstance(assets.get(key),str) or not assets[key]:
                        raise RuntimeError(f'{stem}: asset {key} ausente para {sid}')
                if ds == 'urbansyn' and not row.get('revision'):
                    raise RuntimeError(f'{stem}: revision fijada ausente para {sid}')
            ordered_ids = sorted(sm, key=lambda sid: hashlib.sha256(f'2026:{sid}'.encode()).hexdigest())
            manifest_out[stem] = [{k:v for k,v in sm[sid].items()
                                   if k not in ('semantic_pixels','instance_counts')}
                                  for sid in ordered_ids]
            index_out[stem] = [am[sid] for sid in ordered_ids]
            checks.append({'split':stem,'manifest_rows':len(sem),'instance_rows':len(ins),
                           'index_rows':len(ann),'ids_unique':True,'ids_match':True,
                           'statistics_match':True,'annotation_status_ok':True,
                           'required_assets_present':True})
for ds in ('a2d2','urbansyn'):
    tr={str(x['sample_id']) for x in manifest_out[f'{ds}_train']}
    va={str(x['sample_id']) for x in manifest_out[f'{ds}_val']}
    overlap=tr & va
    if overlap:
        raise RuntimeError(f'{ds}: fuga TRAIN/VAL detectada ({len(overlap)} IDs)')
    checks.append({'dataset':ds,'train_val_overlap':0})

print(json.dumps(checks,indent=2,ensure_ascii=False))

## 4. Persistir manifiestos e índices completos en una carpeta nueva de Drive

No se sobrescribe `ProcesamientoImagenes/urban_data_capacity`, que contiene las 40 muestras de preflight.

In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive/ProcesamientoImagenes')
BASE_FULL_DIR = DRIVE_ROOT/'urban_v1_full_metadata_20261124'

def jsonl_matches(path, expected_rows):
    if not path.is_file():
        return False
    try:
        existing = [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]
        return existing == expected_rows
    except Exception:
        return False

# Reutiliza una carpeta previa solo si los ocho JSONL coinciden exactamente
# con los metadatos recién reconstruidos. Si está incompleta o es distinta,
# crea una carpeta nueva; nunca borra ni sobrescribe los manifiestos existentes.
expected_files = {}
for stem, rows in manifest_out.items():
    expected_files[f'manifests/{stem}.jsonl'] = rows
for stem, rows in index_out.items():
    expected_files[f'annotation_index/{stem}.annotations.jsonl'] = rows

reuse_existing = False
if BASE_FULL_DIR.exists():
    reuse_existing = all(
        jsonl_matches(BASE_FULL_DIR/relative_path, rows)
        for relative_path, rows in expected_files.items()
    )
    if reuse_existing:
        FULL_DIR = BASE_FULL_DIR
        print('Se reutiliza carpeta existente: los ocho JSONL coinciden exactamente.')
    else:
        suffix = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
        FULL_DIR = DRIVE_ROOT/f'urban_v1_full_metadata_{suffix}'
        if FULL_DIR.exists():
            raise RuntimeError(f'La carpeta alternativa {FULL_DIR} también existe; no se sobrescribe. Vuelve a ejecutar esta celda para generar otro sufijo.')
        print('La carpeta existente está incompleta o contiene otros datos; se conserva intacta y se crea:', FULL_DIR)
else:
    FULL_DIR = BASE_FULL_DIR

MANIFEST_DIR = FULL_DIR/'manifests'
INDEX_DIR = FULL_DIR/'annotation_index'
if not reuse_existing:
    for stem, rows in manifest_out.items():
        write_jsonl(MANIFEST_DIR/f'{stem}.jsonl',rows)
    for stem, rows in index_out.items():
        write_jsonl(INDEX_DIR/f'{stem}.annotations.jsonl',rows)

# Verificación posterior a escritura/reutilización antes de continuar.
for relative_path, rows in expected_files.items():
    if not jsonl_matches(FULL_DIR/relative_path, rows):
        raise RuntimeError(f'Verificación posterior fallida: {FULL_DIR/relative_path}')

file_hashes = {}
for p in sorted(FULL_DIR.rglob('*.jsonl')):
    file_hashes[str(p.relative_to(FULL_DIR))] = hashlib.sha256(p.read_bytes()).hexdigest()
metadata_report = {
    'artifact_id':ARTIFACT_ID,'source_run':RUN_ID,'artifact_sha256':EXPECTED_SHA256,
    'git_head':HEAD,'created_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'expected_rows':{f'{ds}_{sp}':n for (ds,sp),n in EXPECTED.items()},
    'checks':checks,'sha256':file_hashes,'reused_existing_metadata':reuse_existing,
    'test_split_touched':False,'assets_downloaded':False,'training_started':False
}
(FULL_DIR/'reconstruction_report.json').write_text(json.dumps(metadata_report,indent=2,ensure_ascii=False)+chr(10))
print('Metadatos completos verificados en:',FULL_DIR)
print(json.dumps({'rows':{k:len(v) for k,v in manifest_out.items()},
                  'jsonl_files':len(file_hashes),'reused_existing_metadata':reuse_existing,
                  'test_split_touched':False,'assets_downloaded':False},indent=2))

## 5. Ejecutar estimación remota completa (dry-run)

El comando no usa `--max-samples-per-split` ni `--require-instances`: estima los 3.751 registros y todos sus assets requeridos. `--dry-run` evita descargar imágenes/máscaras.

In [ ]:
import importlib.util, subprocess, json, shutil
missing = [name for name in ('boto3','huggingface_hub') if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable,'-m','pip','install','-q',*missing],check=True)

DRY_DIR = Path('/content/urban_v1_full_dry_run')
if DRY_DIR.exists():
    shutil.rmtree(DRY_DIR)
cmd = [sys.executable,'scripts/materialize_urban_experiment_data.py',
       '--manifest-dir',str(MANIFEST_DIR),
       '--annotation-index-dir',str(INDEX_DIR),
       '--output-dir',str(DRY_DIR),
       '--dry-run']
run = subprocess.run(cmd,cwd=WORKDIR,text=True,capture_output=True)
print(run.stdout[-20000:])
if run.returncode:
    print(run.stderr[-12000:])
    raise RuntimeError('Falló el dry-run completo. No se descargaron assets.')
report_path = DRY_DIR/'materialization_preflight.json'
if not report_path.is_file():
    raise RuntimeError('No se generó materialization_preflight.json.')
estimate = json.loads(report_path.read_text())
if estimate.get('status') != 'PREFLIGHT_PASS' or estimate.get('dry_run') is not True:
    raise RuntimeError('El materializador no confirmó PREFLIGHT_PASS/dry_run.')
if estimate.get('test_touched') is not False or estimate.get('training_started') is not False:
    raise RuntimeError('El reporte no confirma aislamiento de TEST y entrenamiento.')
expected_samples = sum(EXPECTED.values())
if sum(x['selected_samples'] for x in estimate['selection_validation']) != expected_samples:
    raise RuntimeError('El dry-run no cubrió los 3.751 registros esperados.')
if any(x['manifest_samples'] != EXPECTED[(x['dataset'],x['split'])] for x in estimate['selection_validation']):
    raise RuntimeError('El dry-run detectó conteos distintos del protocolo.')
shutil.copy2(report_path,FULL_DIR/'materialization_preflight_full.json')
print(json.dumps({'status':estimate['status'],
                  'samples':sum(x['selected_samples'] for x in estimate['selection_validation']),
                  'assets':estimate['asset_count'],
                  'estimated_remote_asset_bytes':estimate['estimated_remote_asset_bytes'],
                  'estimated_remote_asset_gib':estimate['estimated_remote_asset_gib'],
                  'test_touched':estimate['test_touched'],
                  'training_started':estimate['training_started'],
                  'report_in_drive':str(FULL_DIR/'materialization_preflight_full.json')},indent=2))

## 6. Materializar TRAIN/VAL completos en el disco temporal de Colab

La estimación anterior cubre todos los assets requeridos de los 3.751 registros. Se comprueba espacio libre con un margen del 15%. No se copia ningún dataset al teléfono ni se escribe en el repositorio.

In [ ]:
import math, shutil, subprocess, sys, json
estimated_bytes = int(estimate["estimated_remote_asset_bytes"])
free_bytes = shutil.disk_usage("/content").free
required_bytes = math.ceil(estimated_bytes * 1.15)
print({"estimated_asset_gib": round(estimated_bytes/1024**3, 3),
       "free_disk_gib": round(free_bytes/1024**3, 3),
       "required_with_15pct_margin_gib": round(required_bytes/1024**3, 3)})
if free_bytes < required_bytes:
    raise RuntimeError("Espacio temporal insuficiente en Colab. No se ha iniciado la descarga de assets.")
DATA_DIR = Path("/content/urban_v1_training_data")
if DATA_DIR.exists():
    # Se conserva y reutiliza el contenido; el materializador valida tamaños y vuelve
    # a escribir los manifiestos locales a partir de la selección completa.
    print("Se reutiliza el directorio temporal existente:", DATA_DIR)
cmd = [sys.executable, "scripts/materialize_urban_experiment_data.py",
       "--manifest-dir", str(MANIFEST_DIR),
       "--annotation-index-dir", str(INDEX_DIR),
       "--output-dir", str(DATA_DIR),
       "--max-total-bytes", str(estimated_bytes)]
materialize = subprocess.run(cmd, cwd=WORKDIR, text=True, capture_output=True)
print(materialize.stdout[-16000:])
if materialize.returncode:
    print(materialize.stderr[-12000:])
    raise RuntimeError("Falló la materialización real. No se iniciará entrenamiento.")
material_report_path = DATA_DIR/"materialization_report.json"
if not material_report_path.is_file():
    raise RuntimeError("No existe materialization_report.json.")
material_report = json.loads(material_report_path.read_text())
if material_report.get("status") != "PASS" or material_report.get("test_touched") is not False:
    raise RuntimeError("Materialización sin PASS o sin confirmación de aislamiento de TEST.")
if sum(x["samples"] for x in material_report["datasets"]) != 3751:
    raise RuntimeError("La materialización no contiene los 3.751 registros esperados.")
print("Materialización completa verificada:", material_report_path)

## 7. Validar archivos locales, manifiestos y dependencias de entrenamiento

Se comprueban los conteos completos, IDs únicos/alineados, rutas locales de imágenes y etiquetas, y el archivo class_list de A2D2. Después se instalan únicamente las dependencias que faltan.

In [ ]:
import importlib.util, hashlib, json, subprocess, sys
missing = [x for x in ("pycocotools",) if importlib.util.find_spec(x) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
from pathlib import Path

EXPECTED_ROWS = {("a2d2","train"):1000, ("a2d2","val"):1000,
                 ("urbansyn","train"):1000, ("urbansyn","val"):751}
LOCAL_MANIFEST = DATA_DIR/"local_manifests"
LOCAL_INDEX = DATA_DIR/"local_annotation_index"
required_fields = {"a2d2":("image","semantic","instance"),
                   "urbansyn":("image","semantic","panoptic")}
local_audit = []
for (ds, split), expected_n in EXPECTED_ROWS.items():
    mp = LOCAL_MANIFEST/f"{ds}_{split}.jsonl"
    ip = LOCAL_INDEX/f"{ds}_{split}.annotations.jsonl"
    rows = [json.loads(x) for x in mp.read_text().splitlines() if x.strip()]
    anns = [json.loads(x) for x in ip.read_text().splitlines() if x.strip()]
    ids = [str(x["sample_id"]) for x in rows]
    ann_ids = [str(x["sample_id"]) for x in anns]
    if len(rows) != expected_n or len(anns) != expected_n:
        raise RuntimeError(f"{ds}/{split}: conteo local {len(rows)}/{len(anns)}, esperado {expected_n}")
    if len(set(ids)) != expected_n or ids != ann_ids:
        raise RuntimeError(f"{ds}/{split}: IDs duplicados o desalineados")
    missing_paths = []
    for i, row in enumerate(rows):
        for key in required_fields[ds]:
            p = Path(row[key])
            if not p.is_file():
                missing_paths.append({"row":i,"field":key,"path":str(p)})
    if missing_paths:
        raise RuntimeError(f"{ds}/{split}: faltan {len(missing_paths)} assets; muestra={missing_paths[:3]}")
    if any(x.get("annotation_status","ok") != "ok" for x in anns):
        raise RuntimeError(f"{ds}/{split}: anotaciones no OK")
    local_audit.append({"dataset":ds,"split":split,"rows":len(rows),
                        "assets_exist":True,"ids_aligned":True})
if not (DATA_DIR/"a2d2"/"class_list.json").is_file():
    raise RuntimeError("Falta a2d2/class_list.json")
for ds in ("a2d2","urbansyn"):
    tr={str(x["sample_id"]) for x in (json.loads(s) for s in (LOCAL_MANIFEST/f"{ds}_train.jsonl").read_text().splitlines() if s.strip())}
    va={str(x["sample_id"]) for x in (json.loads(s) for s in (LOCAL_MANIFEST/f"{ds}_val.jsonl").read_text().splitlines() if s.strip())}
    if tr & va:
        raise RuntimeError(f"Fuga TRAIN/VAL detectada en {ds}")
print(json.dumps(local_audit, indent=2))

## 8. Piloto GPU obligatorio: 10 batches por configuración

El protocolo exige medir las cuatro combinaciones antes del entrenamiento completo. Los pilotos usan modelos recién inicializados y sus actualizaciones se descartan; el entrenamiento completo vuelve a crear modelos desde cero. Si cualquier piloto falla, la notebook no inicia el entrenamiento completo.

In [ ]:
import gc, itertools, math, traceback, time
import torch
from scripts.urban_experiment_factory import create_experiment
from src.urban_experiment_runner import _move_targets
from src.urban_ontology import load_urban_ontology, semantic_name_to_id

CONFIG_PATH = WORKDIR/"configs/urban_experiment_v1.json"
base_config = json.loads(CONFIG_PATH.read_text())
EXPERIMENTS = base_config["experiments"]
RUN_TRAINING = True
if not RUN_TRAINING:
    raise RuntimeError("RUN_TRAINING está desactivado.")
DEVICE = torch.device("cuda")
PILOT_RESULTS = []

def pilot_experiment(spec):
    cfg = dict(spec)
    cfg.update({"data_dir":str(DATA_DIR),
                "annotation_index_dir":str(LOCAL_INDEX),
                "amp":False, "num_workers":0})
    built = create_experiment(cfg)
    model = built["model"].to(DEVICE)
    optimizer = built["optimizer"]
    loader = built["train_loader"]
    task = cfg["task"]
    accum = int(cfg.get("gradient_accumulation_steps",1))
    model.train()
    optimizer.zero_grad(set_to_none=True)
    torch.cuda.reset_peak_memory_stats()
    started = time.time()
    losses=[]
    for batch_no, batch in enumerate(itertools.islice(loader, 10), start=1):
        if task == "semantic":
            images, masks = batch
            images = images.to(DEVICE, non_blocking=True)
            masks = masks.to(DEVICE, dtype=torch.long, non_blocking=True)
            logits = model(images)["out"]
            loss = torch.nn.functional.cross_entropy(
                logits, masks,
                weight=built["class_weights"].to(DEVICE) if built["class_weights"] is not None else None,
                ignore_index=int(cfg.get("ignore_index",255)))
        else:
            images, targets = batch
            images = [x.to(DEVICE, non_blocking=True) for x in images]
            targets = _move_targets(targets, DEVICE)
            loss_dict = model(images, targets)
            if not loss_dict:
                raise RuntimeError("Mask R-CNN devolvió un diccionario de pérdidas vacío.")
            loss = sum(loss_dict.values())
        if loss.ndim != 0 or not torch.isfinite(loss):
            raise FloatingPointError(f"{cfg['experiment_id']}: loss no finita en batch {batch_no}")
        (loss / accum).backward()
        losses.append(float(loss.detach().cpu()))
        if batch_no % accum == 0:
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            if not torch.isfinite(grad_norm):
                raise FloatingPointError("Norma de gradiente no finita.")
            if not all(p.grad is None or torch.isfinite(p.grad).all() for p in model.parameters()):
                raise FloatingPointError("Gradientes no finitos.")
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
    if len(losses) != 10:
        raise RuntimeError(f"{cfg['experiment_id']}: solo se obtuvieron {len(losses)} batches.")
    result = {"experiment_id":cfg["experiment_id"],"batches":len(losses),
              "optimizer_steps":10//accum,"mean_loss":sum(losses)/len(losses),
              "seconds":round(time.time()-started,2),
              "peak_cuda_gib":round(torch.cuda.max_memory_allocated()/1024**3,3),
              "finite_losses":True,"finite_gradients":True}
    del built, model, optimizer, loader
    gc.collect(); torch.cuda.empty_cache()
    return result

PILOT_ERROR = None
try:
    for spec in EXPERIMENTS:
        print("Piloto:", spec["experiment_id"], flush=True)
        result = pilot_experiment(spec)
        PILOT_RESULTS.append(result)
        print(json.dumps(result), flush=True)
except Exception as exc:
    PILOT_ERROR = {"type":type(exc).__name__,"message":str(exc),"traceback":traceback.format_exc()[-10000:]}
    print("PILOTO FALLIDO:", json.dumps(PILOT_ERROR, indent=2))
if PILOT_ERROR or len(PILOT_RESULTS) != 4:
    raise RuntimeError("Piloto incompleto: entrenamiento completo bloqueado. Revisa PILOT_ERROR.")

## 9. Entrenamiento de los cuatro experimentos con reanudación

Cada experimento dispone de hasta 90 minutos por sesión (configuración del protocolo), guarda checkpoints atómicos al terminar cada época y puede reanudarse en una sesión Colab posterior. Los resultados permanecen en Drive privado. No se evalúa TEST.

In [ ]:
import os, json, hashlib, time, traceback, gc
from pathlib import Path
from src.urban_experiment_runner import run_experiment

DRIVE_RESULTS = Path("/content/drive/MyDrive/ProcesamientoImagenes/experimentos/urban-exp-v1")
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
TRAINING_RESULTS = []
TRAINING_ERRORS = []

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""):
            h.update(chunk)
    return h.hexdigest()

for spec in EXPERIMENTS:
    exp_id = spec["experiment_id"]
    print("\n===== INICIANDO", exp_id, "=====", flush=True)
    try:
        cfg = dict(spec)
        cfg.update({"data_dir":str(DATA_DIR),
                    "annotation_index_dir":str(LOCAL_INDEX),
                    "amp":False, "num_workers":0})
        out = DRIVE_RESULTS/exp_id
        (out/"checkpoints").mkdir(parents=True, exist_ok=True)
        (out/"metrics").mkdir(parents=True, exist_ok=True)
        (out/"logs").mkdir(parents=True, exist_ok=True)
        last = out/"checkpoints"/"last.pt"
        best = out/"checkpoints"/"best.pt"
        metrics_file = out/"metrics"/"history.jsonl"
        # El runner escribe sus checkpoints en output_dir raíz; se normalizan
        # después a la estructura checkpoints/ conservando el archivo original.
        built = create_experiment(cfg)
        resume = last if last.is_file() else (out/"last.pt" if (out/"last.pt").is_file() else None)
        run = run_experiment(**built, output_dir=out, config=cfg,
                             resume_from=resume, device=DEVICE)
        # Copiar los checkpoints generados por el runner a la carpeta estable.
        import shutil as _shutil
        if (out/"last.pt").is_file():
            _shutil.copy2(out/"last.pt", last)
        if (out/"best.pt").is_file():
            _shutil.copy2(out/"best.pt", best)
        if (out/"metrics.jsonl").is_file():
            _shutil.copy2(out/"metrics.jsonl", metrics_file)
        state = torch.load(last if last.is_file() else out/"last.pt",
                           map_location="cpu", weights_only=False)
        finished = (int(state.get("epoch",0)) >= int(cfg["max_epochs"]) or
                    int(state.get("bad_epochs",0)) >= int(cfg["patience"]))
        summary = {
            "experiment_id":exp_id,
            "status":"COMPLETED" if finished else "PAUSED_RESUMABLE",
            "epochs_completed_total":int(state.get("epoch",0)),
            "max_epochs":int(cfg["max_epochs"]),
            "best_metric":state.get("best_metric"),
            "bad_epochs":int(state.get("bad_epochs",0)),
            "config":cfg,
            "config_sha256":hashlib.sha256(json.dumps(cfg,sort_keys=True,separators=(",",":")).encode()).hexdigest(),
            "git_head":HEAD,
            "manifest_sha256":{p.name:sha256_file(p) for p in sorted(LOCAL_MANIFEST.glob("*.jsonl"))},
            "annotation_index_sha256":{p.name:sha256_file(p) for p in sorted(LOCAL_INDEX.glob("*.jsonl"))},
            "test_evaluated":False,
            "pilot":next(x for x in PILOT_RESULTS if x["experiment_id"]==exp_id),
            "runner_status":run.get("status"),
            "runner_epochs_this_session":run.get("epochs_completed")
        }
        (out/"run_summary.json").write_text(json.dumps(summary,indent=2,ensure_ascii=False)+"\n")
        TRAINING_RESULTS.append(summary)
        print(json.dumps({"experiment_id":exp_id,"status":summary["status"],
                          "epochs_completed_total":summary["epochs_completed_total"],
                          "best_metric":summary["best_metric"]},indent=2),flush=True)
        del built, run, state
        gc.collect(); torch.cuda.empty_cache()
        if not finished:
            print("Límite de tiempo de esta sesión alcanzado; el checkpoint permite reanudar.",flush=True)
            break
    except Exception as exc:
        TRAINING_ERRORS.append({"experiment_id":exp_id,"type":type(exc).__name__,
                                "message":str(exc),"traceback":traceback.format_exc()[-12000:]})
        print("ERROR EN",exp_id,":",traceback.format_exc()[-12000:],flush=True)
        break

print(json.dumps({"completed_or_paused":len(TRAINING_RESULTS),
                  "errors":TRAINING_ERRORS},indent=2,ensure_ascii=False))

## 10. Resumen final en un único bloque

Ejecuta esta celda incluso si el entrenamiento se detuvo con un error. El bloque indica qué experimentos terminaron, cuáles quedaron reanudables y qué error ocurrió. No incluye credenciales ni datos personales.

In [ ]:
payload = {
    "status":"ERROR" if TRAINING_ERRORS else (
        "COMPLETED" if len(TRAINING_RESULTS)==4 and all(x["status"]=="COMPLETED" for x in TRAINING_RESULTS)
        else "PAUSED_OR_PARTIAL"),
    "git_head":HEAD,
    "gpu":{"name":GPU_NAME,"vram_gib":round(GPU_VRAM_GIB,2)},
    "run_training":RUN_TRAINING,
    "metadata_dir":str(FULL_DIR),
    "data_dir":str(DATA_DIR),
    "materialization_status":material_report.get("status"),
    "materialized_samples":sum(x["samples"] for x in material_report["datasets"]),
    "pilot_results":PILOT_RESULTS,
    "pilot_error":PILOT_ERROR,
    "training_results":TRAINING_RESULTS,
    "training_errors":TRAINING_ERRORS,
    "results_dir":str(DRIVE_RESULTS),
    "test_split_touched":False,
    "test_evaluated":False
}
payload_path = DRIVE_RESULTS/"training_continuity_summary.json"
payload_path.write_text(json.dumps(payload,indent=2,ensure_ascii=False)+"\n")
print("===== BLOQUE_CONTINUIDAD_INICIO =====")
print(json.dumps(payload,indent=2,ensure_ascii=False))
print("===== BLOQUE_CONTINUIDAD_FIN =====")
print("Resumen persistente:",payload_path)